# Water meter YOLO training

Before running: enable GPU and Internet in Kaggle Settings, then add the uploaded zip as a Notebook Input.

In [ ]:
# Cell 1: install and configure
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'pyyaml'])

import yaml
from ultralytics import YOLO

EPOCHS = 50
IMAGE_SIZE = 640
BATCH = 16  # Change to 8 if GPU memory is insufficient.
MODEL_NAME = 'yolo11n.pt'
RUN_NAME = 'water_meter_yolo11n'

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
EXTRACT_DIR = WORK_ROOT / 'water_meter_dataset'
RUNS_DIR = WORK_ROOT / 'runs' / 'water_meter'

In [ ]:
# Cell 2: find the input. Kaggle may keep the zip or automatically unpack it.
zip_files = sorted(INPUT_ROOT.rglob('*.zip'))
yaml_files = sorted(INPUT_ROOT.rglob('data.yaml'))

if zip_files:
    INPUT_TYPE = 'zip'
    SOURCE_PATH = zip_files[0]
elif yaml_files:
    INPUT_TYPE = 'folder'
    SOURCE_PATH = yaml_files[0]
else:
    available = [str(path) for path in INPUT_ROOT.iterdir()]
    raise FileNotFoundError(f'No zip or data.yaml found. Available inputs: {available}')

print('Input type:', INPUT_TYPE)
print('Using:', SOURCE_PATH)

In [ ]:
# Cell 3: copy the dataset to the writable area and make paths work on Kaggle
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

if INPUT_TYPE == 'zip':
    EXTRACT_DIR.mkdir(parents=True)
    with zipfile.ZipFile(SOURCE_PATH) as archive:
        archive.extractall(EXTRACT_DIR)
else:
    shutil.copytree(SOURCE_PATH.parent, EXTRACT_DIR)

yaml_candidates = sorted(EXTRACT_DIR.rglob('data.yaml'))
if not yaml_candidates:
    raise FileNotFoundError('data.yaml was not found inside the zip.')

data_yaml = yaml_candidates[0]
DATASET_DIR = data_yaml.parent
config = yaml.safe_load(data_yaml.read_text(encoding='utf-8'))

missing = {'train', 'val', 'names'} - set(config)
if missing:
    raise ValueError(f'data.yaml is missing: {sorted(missing)}')

normalized_config = {
    'path': str(DATASET_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'names': config['names'],
    'nc': config.get('nc', len(config['names'])),
}
data_yaml.write_text(yaml.safe_dump(normalized_config, sort_keys=False), encoding='utf-8')

print(data_yaml.read_text())

In [ ]:
# Cell 4: verify the extracted dataset before training
for split in ['train', 'valid']:
    for kind in ['images', 'labels']:
        folder = DATASET_DIR / split / kind
        print(f'{split}/{kind}: {len(list(folder.glob("*")))} files')

print('Classes:', normalized_config['names'])

In [ ]:
# Cell 5: train YOLO
model = YOLO(MODEL_NAME)
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    device=0,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    patience=10,
    seed=42,
)

In [ ]:
# Cell 6: evaluate the best model and create visual predictions
best_weights = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
if not best_weights.exists():
    raise FileNotFoundError(f'best.pt not found: {best_weights}')

best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(data_yaml), imgsz=IMAGE_SIZE, device=0)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

sample_images = sorted((DATASET_DIR / 'valid' / 'images').glob('*'))[:10]
best_model.predict(
    source=[str(image) for image in sample_images],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    save=True,
    project=str(WORK_ROOT / 'predictions'),
    name='validation_samples',
)

In [ ]:
# Cell 7: package the model, curves, and metrics for download
archive_path = shutil.make_archive(
    str(WORK_ROOT / 'water_meter_yolo_output'),
    'zip',
    root_dir=RUNS_DIR / RUN_NAME,
)
print('Best weights:', best_weights)
print('Download this file from Kaggle Output:', archive_path)